In [11]:
# app.py
import os
import json
from typing import List, Tuple
import redis
from pydantic import BaseModel

from config import REDIS_URL, MAX_HISTORY_MESSAGES, gemini_chat_model, API_KEY
from milvus_vectorstore import get_vectorstore

# LangChain imports
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
# assume you already have your Milvus vectorstore instance

In [2]:
# docker-compose up -d


In [3]:

r = redis.Redis.from_url(REDIS_URL, decode_responses=True)
vectorstore = get_vectorstore()   # <-- set your Milvus vectorstore instance here


In [4]:

# ----- Pydantic models -----
class CreateSessionResp(BaseModel):
    session_id: str

class ChatReq(BaseModel):
    session_id: str
    question: str

class ChatResp(BaseModel):
    answer: str
    sources: List[dict]

In [5]:
# ----- Helpers for Redis history -----
# Key patterns:
# "session:{session_id}:history" -> Redis list of JSON items: [{"role":"user","text":...}, {"role":"assistant","text":..., "sources":[...]}, ...]
def _history_key(session_id: str) -> str:
    return f"session:{session_id}:history"

def load_recent_history(session_id: str, n: int = MAX_HISTORY_MESSAGES) -> List[dict]:
    # expects Redis list of JSON strings: {"role":"user"/"assistant", "text":"..."}
    key = _history_key(session_id)
    raw = r.lrange(key, -n, -1) if r.exists(key) else []
    return [json.loads(x) for x in raw]

def append_history_item(session_id: str, role: str, text: str, sources: List[dict] = None):
    item = {"role": role, "text": text}
    if sources:
        item["sources"] = sources
    r.rpush(_history_key(session_id), json.dumps(item))
    # optionally set TTL: r.expire(_history_key(session_id), TTL_SECONDS)


In [6]:
retriever = vectorstore.as_retriever(search_type="similarity",
                                        search_kwargs={"k": 6})  # tune k

In [8]:
# Build LLM and chain
llm = ChatGoogleGenerativeAI(model=gemini_chat_model, google_api_key=os.environ[API_KEY])  # your Gemini chat model

In [9]:
# We'll use a ConversationBufferMemory and populate it from Redis
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

C:\Users\USER\AppData\Local\Temp\ipykernel_17280\3495178477.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [12]:
# Redis-backed "memory-like" helper (mimics old ConversationBufferMemory enough for drop-in)
class RedisConversationMemory:
    def __init__(self, session_id: str, max_messages: int = MAX_HISTORY_MESSAGES):
        self.session_id = session_id
        self.max_messages = max_messages

    def load(self) -> List[dict]:
        return load_recent_history(self.session_id, n=self.max_messages)

    def get_chat_history_pairs(self) -> List[Tuple[str, str]]:
        """
        Return list of (user_text, assistant_text) tuples expected by many ConversationalRetrievalChains.
        If there are stray messages they will be handled reasonably:
          - user followed by assistant -> paired
          - user without assistant -> (user, "")
          - assistant without preceding user -> ("", assistant)
        """
        recent = self.load()
        pairs = []
        i = 0
        n = len(recent)
        while i < n:
            if recent[i]["role"] == "user":
                user_text = recent[i]["text"]
                assistant_text = ""
                # find next assistant message immediately following (not skipping other users)
                if i + 1 < n and recent[i + 1]["role"] == "assistant":
                    assistant_text = recent[i + 1]["text"]
                    i += 2
                else:
                    i += 1
                pairs.append((user_text, assistant_text))
            else:
                # assistant without preceding user -> preserve it as ("", assistant)
                pairs.append(("", recent[i]["text"]))
                i += 1
        return pairs

    def add_user_message(self, text: str):
        append_history_item(self.session_id, "user", text)

    def add_assistant_message(self, text: str, sources: List[dict] = None):
        append_history_item(self.session_id, "assistant", text, sources=sources)

In [15]:
session_id=1

In [16]:
# create memory helper for this session
mem = RedisConversationMemory(session_id=session_id, max_messages=MAX_HISTORY_MESSAGES)


In [17]:
# Build retriever & LLM as you did previously (assuming 'retriever' and 'llm' exist)
# conv_chain WITHOUT ConversationBufferMemory
conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=None,  # do not pass deprecated memory object
    return_source_documents=True,
)

In [ ]:
# Build chat_history pairs and call chain
chat_history_pairs = mem.get_chat_history_pairs()  # list of tuples: [(user, assistant), ...]
result = conv_chain({"question": question, "chat_history": chat_history_pairs})